In [240]:
# !pip install yfinance gspread oauth2client

In [241]:
import yfinance as yf
import pandas as pd
import gspread
from oauth2client.service_account import ServiceAccountCredentials

In [242]:
# =============================
# GOOGLE SHEETS CONNECTION
# =============================
scope = ["https://spreadsheets.google.com/feeds",
         "https://www.googleapis.com/auth/drive"]

creds = ServiceAccountCredentials.from_json_keyfile_name(
    "../credentials.json", scope)

client = gspread.authorize(creds)
sheet = client.open("AlgoTradingLogs").sheet1

In [243]:
# STOCK LIST
stocks = [
    "RELIANCE.NS",
    "TCS.NS",
    "INFY.NS",
    "HDFCBANK.NS",
    "ICICIBANK.NS",
    "SBIN.NS",
    "HINDUNILVR.NS",
    "ITC.NS",
    "LT.NS",
    "KOTAKBANK.NS",
    "YESBANK.NS", "IDEA.NS"
]

In [244]:
for stock in stocks:

    df = yf.download(stock, start="2000-01-01", end="2026-01-01")
    df = df.reset_index()

    if isinstance(df.columns, pd.MultiIndex):
        df.columns = [col[0] for col in df.columns]

    df['Date'] = pd.to_datetime(df['Date'])

    # INDICATORS
    df['SMA_20'] = df['Close'].rolling(20).mean()
    df['SMA_50'] = df['Close'].rolling(50).mean()

    # SIGNAL
    df['Signal'] = "HOLD"
    df.loc[df['SMA_20'] > df['SMA_50'], 'Signal'] = "BUY"
    df.loc[df['SMA_20'] < df['SMA_50'], 'Signal'] = "SELL"

    # LATEST
    latest = df.iloc[-1]

    date = str(latest['Date'])[:10]
    price = round(float(latest['Close']), 2)
    signal = latest['Signal']

    stock_name = stock.replace(".NS", "")

    print(date, stock_name, price, signal)

    # LOGGING INSIDE LOOP
    quantity = 10

    data = sheet.get_all_values()
    rows = data[1:]

    new_entry = [str(date), stock_name, signal, str(price)]

    rows_clean = [[str(col).strip() for col in row[:4]] for row in rows]

    if new_entry in rows_clean:
        print(f"{stock_name}: Duplicate — skipped")

    else:
        if signal == "BUY":
            sheet.append_row([date, stock_name, signal, price, quantity, ""])
            print(f"{stock_name}: BUY logged")

        elif signal == "SELL":

            buy_price = None

            for row in reversed(rows):
                if row[1] == stock_name and row[2] == "BUY":
                    buy_price = float(row[3])
                    break

            if buy_price:
                pnl = (price - buy_price) * quantity
            else:
                pnl = 0

            sheet.append_row([date, stock_name, signal, price, quantity, pnl])
            print(f"{stock_name}: SELL logged with PnL = {pnl}")

[*********************100%***********************]  1 of 1 completed


2025-12-31 RELIANCE 1570.4 BUY
RELIANCE: Duplicate — skipped ❌


[*********************100%***********************]  1 of 1 completed


2025-12-31 TCS 3148.96 BUY
TCS: Duplicate — skipped ❌


[*********************100%***********************]  1 of 1 completed


...

2025-12-31 KOTAKBANK 440.22 BUY
KOTAKBANK: Duplicate — skipped ❌


[*********************100%***********************]  1 of 1 completed


2025-12-31 YESBANK 21.6 SELL
YESBANK: SELL logged with PnL = -264.90000000000003 ✅


[*********************100%***********************]  1 of 1 completed


2025-12-31 IDEA 10.76 BUY
IDEA: BUY logged ✅
